# HW1 — Monte Carlo

**Course:** Reinforcement Learning — IE University Madrid  
**Instructor:** Jaume Manero  
**Author:** Elad Moshe  
**Program:** MCSBT  
**Date:** May 2026  

---

## Section 6 — Final Questions



---
### Q1 — Comparative Table

Results across all activities. "Episodes to 95%" is the rolling-window episode at which the training success rate first crossed 95%; "—" means the method did not reach that threshold within the training budget.

| Environment | Method | Episodes | Train time (s) | Episodes to 95% | Final train rate | Greedy eval |
|-------------|--------|----------|---------------|-----------------|-----------------|-------------|
| FrozenLake 4×4 | Monte Carlo | 10,000 | ~1 | — | 99.2% | — |
| FrozenLake 4×4 | Monte Carlo | 20,000 | 1.5 | 8,921 | 99.0% | — |
| FrozenLake 4×4 | TD(0) | 20,000 | 4.3 | 8,132 | 98.0% | **100%** |
| FrozenLake 8×8 | Monte Carlo | 50,000 | ~5 | — | 97.8% | — |
| Volcano | Monte Carlo | 2,000,000 | 201.1 | — | 92.3% | **100%** |
| Taxi | Monte Carlo | 200,000 | 47.7 | — | 100.0% | **100%** |



---
### Q2 — Which Method Shows Better Convergence?

**TD(0) converges faster in episodes; MC converges more cleanly in wall-clock time.**

On FrozenLake 4×4 with identical hyperparameters, TD(0) crossed 95% at episode 8,132 versus 8,921 for MC — roughly 9% fewer episodes. The reason is that TD updates at every step: a 6-step trajectory to the goal produces 6 independent $V$ updates, propagating reward information backwards immediately. MC waits until the episode ends and then does a single backward pass, so the same trajectory produces one update per state visited.

However, MC's per-episode update is much cheaper to compute than TD's per-step loop, which is why MC's wall-clock time (1.5 s) was almost 3× faster than TD(0)'s (4.3 s) for the same 20,000 episodes. On longer-horizon tasks — such as Taxi (200 steps/episode) or Volcano — this per-step overhead becomes negligible relative to the sample-efficiency gain, making TD better option.

---
### Q3 — `is_slippery=True` in FrozenLake

The cell below runs the same MC Control algorithm from Activity 2 on both the non-slippery and slippery variants of FrozenLake 4×4, using identical hyperparameters, and reports the final success rate.

In [1]:
import numpy as np
import gymnasium as gym
import time

def run_episode(env, Q, epsilon, max_steps=200):
    state, _ = env.reset()
    episode = []
    for _ in range(max_steps):
        action = (env.action_space.sample() if np.random.random() < epsilon
                  else int(np.argmax(Q[state])))
        next_state, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break
    return episode

def mc_control(env, n_episodes=20_000, alpha=0.05, gamma=0.99,
               eps_start=1.0, eps_decay=0.9997, eps_min=0.01):
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    epsilon = eps_start
    wins = []
    for _ in range(n_episodes):
        episode = run_episode(env, Q, epsilon)
        G = 0.0
        for s, a, r in reversed(episode):
            G = r + gamma * G
            Q[s, a] += alpha * (G - Q[s, a])
        epsilon = max(eps_min, epsilon * eps_decay)
        wins.append(1 if episode[-1][2] > 0 else 0)
    return Q, wins

def greedy_eval(env, Q, n_eval=1_000):
    successes = 0
    for _ in range(n_eval):
        state, _ = env.reset()
        done = False
        steps = 0
        while not done and steps < 200:
            action = int(np.argmax(Q[state]))
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1
        if reward > 0:
            successes += 1
    return successes / n_eval

N_EPISODES = 20_000
np.random.seed(42)

results = {}
for slippery, label in [(False, 'Non-slippery'), (True, 'Slippery')]:
    env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=slippery)
    t0 = time.time()
    Q, wins = mc_control(env)
    elapsed = time.time() - t0
    greedy_rate = greedy_eval(env, Q)
    env.close()
    results[label] = {
        'final_train': np.mean(wins[-200:]),
        'greedy':      greedy_rate,
        'time':        elapsed,
    }
    print(f"{label}:  train (last 200) = {np.mean(wins[-200:]):.1%}  "
          f"greedy eval = {greedy_rate:.1%}  time = {elapsed:.1f}s")

Non-slippery:  train (last 200) = 100.0%  greedy eval = 100.0%  time = 1.4s
Slippery:  train (last 200) = 4.5%  greedy eval = 4.9%  time = 1.6s


#### Why `is_slippery=True` Makes the Problem Harder

In the non-slippery environment each action deterministically moves the agent in the chosen direction. With `is_slippery=True` the environment becomes **stochastic**: each action is executed in the intended direction only 1/3 of the time; with probability 2/3 the agent slips to one of the two perpendicular directions instead (Gymnasium's default slip model).

This changes the problem in three important ways:

1. **The optimal policy changes.** A path that is safe under determinism (e.g., walking along an edge adjacent to a hole) becomes risky when slipping is possible. The agent must learn a more conservative policy that avoids states from which a random slip ends the episode in a hole.

2. **The value function is harder to estimate.** With stochastic transitions, $Q(s, a)$ must average over many possible outcomes. The same state-action pair can lead to very different returns depending on which slip occurred, so Monte Carlo estimates have higher variance and require more episodes to converge.

3. **Convergence may plateau below 100%.** Even the optimal stochastic policy cannot guarantee reaching the goal on every episode — some slip sequences are unavoidable. The theoretical best success rate on the 4×4 slippery map is around 70–75%, not 100%.

---
### Q4 — FrozenLake 4×4 vs FrozenLake 8×8

| | FrozenLake 4×4 | FrozenLake 8×8 |
|--|--|--|
| States | 16 | 64 |
| Holes | 4 | 9 |
| Optimal path length | 6 steps | ~14 steps |
| Episodes trained | 10,000 | 50,000 |
| Final train rate | 99.2% | 97.8% |

**How does the larger state space affect learning?**

Three effects compound on the 8×8 map:

1. **Slower exploration.** With 64 states instead of 16, a random policy takes much longer to visit every state enough times for $Q$ estimates to become reliable. The probability that a random episode reaches the goal drops sharply as the grid grows, so positive-reward signal propagates more slowly.

2. **Longer credit assignment.** The optimal path is roughly 14 steps vs 6 on the 4×4. Monte Carlo must back-propagate the return $G_t$ through 14 discounted steps instead of 6, so the learning signal received by early states (near the start) is smaller by a factor of $\gamma^{14} / \gamma^6 = \gamma^8 \approx 0.92$ (with $\gamma=0.99$). Start-state values therefore converge more slowly.

3. **More training episodes required.** The 8×8 map needed 50,000 episodes to reach 97.8% final train rate — comparable to the 99.2% the 4×4 achieved in just 10,000 episodes. Reaching similar performance required 5× more training episodes, reflecting the larger and sparser-reward state space.

---
### Q5 — FrozenLake vs Taxi: Which Is Harder for Monte Carlo?

| | FrozenLake 4×4 | Taxi |
|--|--|--|
| States | 16 | 500 |
| Actions | 4 | 6 |
| Q-table size | 64 | 3,000 |
| Reward structure | sparse (+1 goal only) | dense (pickup +1, dropoff +20, illegal −10, step −1) |
| Start state | fixed | random |
| Episodes to converge | ~10,000 | ~200,000 |
| Final greedy eval | 100% | 100% |

**Taxi is significantly harder for Monte Carlo**, for four reasons:

1. **Larger state space.** Taxi has 500 states (25 taxi positions × 5 passenger locations × 4 destinations) vs 16. The Q-table is 47× larger, so far more episodes are needed to estimate every entry reliably.

2. **Random starting position.** FrozenLake always starts at state 0. Taxi starts at a random state, so each episode visits different (taxi, passenger, destination) combinations. This is good for coverage but means no single state accumulates as many visits per episode, slowing convergence.

3. **Longer episodes and delayed reward.** A Taxi episode requires the agent to navigate to the passenger, pick up, navigate to the destination, and drop off — typically 13+ steps even optimally. The Monte Carlo return $G_t$ must propagate credit backwards over a longer chain, and early actions (navigation) receive a more discounted reward signal.

4. **Richer but also misleading reward.** The −10 penalty for illegal pickup/dropoff slows early learning as the random policy repeatedly triggers penalties. Monte Carlo must average over many episodes to distinguish productive exploration from penalty-triggering actions.

Despite being harder, Taxi is ultimately very solvable with MC: after 200,000 episodes the greedy policy achieves 100% success, mean reward +7.59, and only 13.4 steps per episode — matching the near-optimal path.